# ⚡ Crushing the Engine: Barrier Options & Celery HPC
A Down-and-Out Barrier Option contains a step-function discontinuity. Calculating this accurately for risk management requires an immense volume of trajectories to stabilize the Greeks (Delta/Gamma).

> **API Key Required:** You will need your private key to access the distributed C++ cluster. Get yours at **[prometheusquantengine.com](https://prometheusquantengine.com)**.

### 🏗️ Asynchronous Orchestration
If a payload exceeds **50,000,000 total computational steps**, Prometheus will automatically intercept the request to prevent HTTP timeouts. The web plane instantly delegates the matrix to our isolated C++ Celery Cluster.

Let's price a Barrier Option pushing **252,000,000 steps** (`N = 1,000,000` paths × `M = 252` steps).
* Cost: **1.008 Credits**.
* Execution Mode: Asynchronous Polling.

In [ ]:
import requests
import uuid
import time

API_KEY = "pmt_live_..." # 👈 PASTE YOUR PRIVATE API KEY HERE
BASE_URL = "https://api.prometheusquantengine.com/api/v1/simulations"
TASK_URL = "https://api.prometheusquantengine.com/api/v1/simulations/task"

headers = {
    "X-API-Key": API_KEY,
    "Idempotency-Key": str(uuid.uuid4()),
    "Content-Type": "application/json"
}

payload = {
    "simulation_type": "Barrier",
    "s_0": 100.0,
    "strike": 100.0,
    "volatility": 0.25,
    "time_to_maturity": 1.0,
    "risk_free_rate": 0.05,
    "option_type": "Put",
    "n_simulations": 1000000, # 1 Million Paths
    "m_steps": 252,           # Daily observations
    "barrier_type": "DownAndOut",
    "barrier_level": 85.0
}

print("Dispatching 252,000,000 computational steps to the Prometheus HPC Broker...")
response = requests.post(BASE_URL, json=payload, headers=headers)

if response.status_code == 403:
    print("❌ Error: The public demo key is restricted. Please use your private API key.")
else:
    data = response.json()
    # Notice it returns a 201 Created with a task_id ticket, NOT a fair_value.
    task_id = data.get("task_id")
    print(f"Control Plane: {data.get('message')}")
    print(f"Task Ticket ID: {task_id}\n")
    
    # Long Polling Protocol against the Celery Data Plane
    print("Initiating Asynchronous Polling...")
    while True:
        task_resp = requests.get(f"{TASK_URL}/{task_id}", headers={"X-API-Key": API_KEY})
        task_data = task_resp.json()
        status = task_data.get("status")
        
        if status == "SUCCESS":
            print("\n✅ COMPUTATION COMPLETE")
            print(f"Fair Value:    {task_data.get('fair_value')}")
            print(f"Simulation ID: {task_data.get('simulation_id')}")
            break
        elif status in ["FAILURE", "REVOKED"]:
            print("\n❌ ENGINE FAILURE. Credits automatically refunded to your escrow.")
            break
        
        print(f"[{time.strftime('%H:%M:%S')}] Cluster Status: {status}. Awaiting C++ OpenMP threads...")
        time.sleep(2) # Polling interval

### 🏁 Conclusion
Attempting to process a 252 Million step matrix containing step-function barrier discontinuities inside a native Python/NumPy kernel would lock your machine, bloat your RAM, and take minutes to resolve. 

By leveraging the API, you effectively treated a high-performance AWS distributed cluster as a simple Python function call.

### 🛡️ Audit & Ledgers
Head back to your **[Developer Dashboard](https://prometheusquantengine.com/dashboard)** to see this exact simulation securely recorded in your ledger, audit your consumed compute credits, and deploy production configurations.